# ML Notebook: Cluster Feast Offline Store To Saved Model

This notebook follows the rubric steps directly:

1. Load data from the local Kubernetes cluster offline store through Feast running inside a K8s pod, then merge with the generated label table from Improve the Data Generator.
2. Split data into training set and validation set.
3. Train a task-appropriate binary classification model on the training dataset.
4. Evaluate the model on the validation dataset.
5. Save the model as `.joblib`.


## Setup: paths and imports

In [3]:
from __future__ import annotations

import os
import random
import subprocess
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "apps/data-platform").exists():
            return candidate
    raise RuntimeError("Could not find RecSys-MLops repository root")


def run_command(command: list[str], *, check: bool = True) -> subprocess.CompletedProcess[str]:
    printable = " ".join(command)
    print(f"$ {printable}")
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed with exit code {result.returncode}: {printable}")
    return result


ROOT = find_repo_root(Path.cwd().resolve())
DATA_GENERATOR_SRC = ROOT / "apps/data-platform/data-generator/src"
ML_SYSTEM_SRC = ROOT / "apps/ml-system/src"
sys.path.insert(0, str(DATA_GENERATOR_SRC))
sys.path.insert(0, str(ML_SYSTEM_SRC))

CONFIG_PATH = ROOT / "configs/local/data_generator_drift.yaml"
RUN_ID = "drift_50k_seed42"
RUN_PATH = DATA_GENERATOR_SRC / "output" / RUN_ID
EXPORT_SCRIPT = ROOT / "notebooks/export_cluster_feast_training_data.py"
LABEL_PATH = ROOT / "notebooks/data/generated_labels.parquet"
CLUSTER_TRAINING_DATA_PATH = ROOT / "notebooks/data/cluster_feast_training_dataset.parquet"
BST_SPLIT_DIR = ROOT / "notebooks/data/cluster_bst_split"
MODEL_DIR = ROOT / "notebooks/models"
CHECKPOINT_DIR = MODEL_DIR / "bst_checkpoints"
MODEL_PATH = MODEL_DIR / "feast_cluster_bst_10epoch.joblib"

NAMESPACE = "recsys-dataflow"
EXPORT_POD = "recsys-feast-training-export"
EXPORT_IMAGE = "recsys-dataflow-cli:local"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LABEL_PATH.parent.mkdir(parents=True, exist_ok=True)
BST_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print(f"repo root: {ROOT}")
print(f"K8s namespace: {NAMESPACE}")
print(f"Feast export pod image: {EXPORT_IMAGE}")
print(f"ML system src: {ML_SYSTEM_SRC.relative_to(ROOT)}")
print(f"generator config: {CONFIG_PATH.relative_to(ROOT)}")


repo root: /Users/KHOAI/anhkhoa/RecSys-MLops
K8s namespace: recsys-dataflow
Feast export pod image: recsys-dataflow-cli:local
ML system src: apps/ml-system/src
generator config: configs/local/data_generator_drift.yaml


## Step 1. Load data from cluster offline store through Feast and merge generated labels

This step builds the generated two-column label table (`user_id`, `label`), starts a temporary K8s pod from `recsys-dataflow-cli:local`, runs Feast inside that pod against the local-cluster MinIO offline store, merges the Feast-loaded features with labels inside the pod, and copies the merged training parquet back to this notebook.

### Step 1a. Build generated label table from Improve Data Generator

In [6]:
from config import load_config
from scripts.summarize_drift_label_merge import build_label_rows

generator_config = load_config(str(CONFIG_PATH))
labels_with_debug = pd.DataFrame(build_label_rows(generator_config, RUN_PATH))
label_table = labels_with_debug[["user_id", "label"]].copy()
label_table.to_parquet(LABEL_PATH, index=False)

print(f"label rows: {len(label_table):,}")
print("label distribution:", label_table["label"].value_counts().sort_index().to_dict())
print("label definition: user has >= 1 order on/after", generator_config.drift.drift_start_date)
print(f"label parquet for pod: {LABEL_PATH.relative_to(ROOT)}")
print(label_table.head(10).to_string(index=False))


label rows: 1,000
label distribution: {0: 316, 1: 684}
label definition: user has >= 1 order on/after 2025-07-30
label parquet for pod: notebooks/data/generated_labels.parquet
 user_id  label
       1      0
       2      0
       3      1
       4      0
       5      1
       6      1
       7      1
       8      1
       9      1
      10      1


### Step 1b. Run Feast inside local Kubernetes pod and export all cluster training entities

In [8]:
run_command(["kubectl", "delete", "pod", "-n", NAMESPACE, EXPORT_POD, "--ignore-not-found=true"])
run_command([
    "kubectl",
    "run",
    EXPORT_POD,
    "-n",
    NAMESPACE,
    "--image",
    EXPORT_IMAGE,
    "--image-pull-policy=IfNotPresent",
    "--restart=Never",
    "--env=AWS_ACCESS_KEY_ID=minio",
    "--env=AWS_SECRET_ACCESS_KEY=minio123",
    "--env=AWS_DEFAULT_REGION=us-east-1",
    "--env=MINIO_ENDPOINT=http://data-platform-minio:9000",
    "--env=AWS_ENDPOINT_URL=http://data-platform-minio:9000",
    "--env=S3_ENDPOINT_URL=http://data-platform-minio:9000",
    "--env=AWS_ALLOW_HTTP=true",
    "--env=AWS_EC2_METADATA_DISABLED=true",
    "--command",
    "--",
    "sleep",
    "3600",
])
run_command(["kubectl", "wait", "-n", NAMESPACE, "--for=condition=Ready", f"pod/{EXPORT_POD}", "--timeout=120s"])

run_command(["kubectl", "cp", str(EXPORT_SCRIPT), f"{NAMESPACE}/{EXPORT_POD}:/tmp/export_cluster_feast_training_data.py"])
run_command(["kubectl", "cp", str(LABEL_PATH), f"{NAMESPACE}/{EXPORT_POD}:/tmp/generated_labels.parquet"])
run_command([
    "kubectl",
    "exec",
    "-n",
    NAMESPACE,
    EXPORT_POD,
    "--",
    "python",
    "/tmp/export_cluster_feast_training_data.py",
    "--labels",
    "/tmp/generated_labels.parquet",
    "--output",
    "/tmp/cluster_feast_training_dataset.parquet",
])
run_command(["kubectl", "cp", f"{NAMESPACE}/{EXPORT_POD}:/tmp/cluster_feast_training_dataset.parquet", str(CLUSTER_TRAINING_DATA_PATH)])

print(f"cluster Feast training parquet copied to: {CLUSTER_TRAINING_DATA_PATH.relative_to(ROOT)}")


$ kubectl delete pod -n recsys-dataflow recsys-feast-training-export --ignore-not-found=true
pod "recsys-feast-training-export" deleted from recsys-dataflow namespace

$ kubectl run recsys-feast-training-export -n recsys-dataflow --image recsys-dataflow-cli:local --image-pull-policy=IfNotPresent --restart=Never --env=AWS_ACCESS_KEY_ID=minio --env=AWS_SECRET_ACCESS_KEY=minio123 --env=AWS_DEFAULT_REGION=us-east-1 --env=MINIO_ENDPOINT=http://data-platform-minio:9000 --env=AWS_ENDPOINT_URL=http://data-platform-minio:9000 --env=S3_ENDPOINT_URL=http://data-platform-minio:9000 --env=AWS_ALLOW_HTTP=true --env=AWS_EC2_METADATA_DISABLED=true --command -- sleep 3600
pod/recsys-feast-training-export created

$ kubectl wait -n recsys-dataflow --for=condition=Ready pod/recsys-feast-training-export --timeout=120s
pod/recsys-feast-training-export condition met

$ kubectl cp /Users/KHOAI/anhkhoa/RecSys-MLops/notebooks/export_cluster_feast_training_data.py recsys-dataflow/recsys-feast-training-export:/t

### Step 1c. Load the merged Feast training dataset exported from the pod

In [10]:
dataset = pd.read_parquet(CLUSTER_TRAINING_DATA_PATH)
FEATURE_COLUMNS = [
    "views_30m",
    "carts_30m",
    "purchases_24h",
    "distinct_categories_7d",
    "avg_viewed_price_7d",
    "cart_to_purchase_ratio_7d",
    "last_event_age_seconds",
]
dataset[FEATURE_COLUMNS] = dataset[FEATURE_COLUMNS].fillna(0).astype("float32")
dataset["label"] = dataset["label"].astype("float32")

print(f"merged training dataset shape: {dataset.shape}")
print("label distribution after cluster Feast merge:", dataset["label"].value_counts().sort_index().to_dict())
print(dataset[["user_id", "label", *FEATURE_COLUMNS]].head(12).to_string(index=False))


merged training dataset shape: (9724, 10)
label distribution after cluster Feast merge: {0.0: 2453, 1.0: 7271}
 user_id  label  views_30m  carts_30m  purchases_24h  distinct_categories_7d  avg_viewed_price_7d  cart_to_purchase_ratio_7d  last_event_age_seconds
     897    1.0        1.0        0.0            0.0                     1.0             1.000000                        0.0                     0.0
     897    1.0        2.0        0.0            0.0                     2.0             1.000000                        0.0                     0.0
     897    1.0        3.0        0.0            0.0                     3.0             1.333333                        0.0                     0.0
     657    1.0        1.0        0.0            0.0                     1.0             2.000000                        0.0                     0.0
     657    1.0        2.0        0.0            0.0                     2.0             2.000000                        0.0                    

## Step 2. Split data into training set and validation set

In [12]:
import json
from sklearn.model_selection import train_test_split

ITEM_NUM = 22_700
CATEGORY_NUM = 30
BRAND_NUM = 740
PRICE_BUCKET_NUM = 10
TIME_BUCKET_NUM = 9
EVENT_TYPE_NUM = 3
MAX_HISTORY_LEN = 5


def bounded_token(value: int, upper: int) -> int:
    return int(value % (upper - 1)) + 1


def timestamp_to_seconds(value) -> int:
    return int(pd.Timestamp(value).timestamp())


def to_bst_record(row: pd.Series, row_index: int) -> dict:
    views = int(row["views_30m"])
    carts = int(row["carts_30m"])
    purchases = int(row["purchases_24h"])
    user_id = int(row["user_id"])
    history_len = max(1, min(MAX_HISTORY_LEN, views + carts + purchases + 1))

    base_item = bounded_token(user_id * 37 + row_index, ITEM_NUM)
    base_category = bounded_token(int(row["distinct_categories_7d"]) + user_id, CATEGORY_NUM)
    base_brand = bounded_token(user_id * 13 + row_index, BRAND_NUM)
    base_price = bounded_token(int(row["avg_viewed_price_7d"]) + int(row["views_30m"]), PRICE_BUCKET_NUM)
    base_time = bounded_token(int(row["last_event_age_seconds"]) // 3600, TIME_BUCKET_NUM)

    hist_item_id = [bounded_token(base_item + offset, ITEM_NUM) for offset in range(history_len)]
    hist_category = [bounded_token(base_category + offset, CATEGORY_NUM) for offset in range(history_len)]
    hist_brand = [bounded_token(base_brand + offset, BRAND_NUM) for offset in range(history_len)]
    hist_price_bucket = [bounded_token(base_price + offset, PRICE_BUCKET_NUM) for offset in range(history_len)]
    hist_time = [bounded_token(base_time + offset, TIME_BUCKET_NUM) for offset in range(history_len)]
    hist_event_type = [2 if offset < carts + purchases else 1 for offset in range(history_len)]

    return {
        "user_id": user_id,
        "hist_item_id": hist_item_id,
        "hist_event_type": hist_event_type,
        "hist_category": hist_category,
        "hist_brand": hist_brand,
        "hist_price_bucket": hist_price_bucket,
        "hist_time": hist_time,
        "target_item_id": bounded_token(base_item + views + 3, ITEM_NUM),
        "target_category": bounded_token(base_category + int(row["distinct_categories_7d"]) + 1, CATEGORY_NUM),
        "target_brand": bounded_token(base_brand + carts + 1, BRAND_NUM),
        "target_price_bucket": bounded_token(base_price + purchases + 1, PRICE_BUCKET_NUM),
        "event_time": timestamp_to_seconds(row["event_timestamp"]),
        "label": int(row["label"]),
    }


def write_jsonl(path: Path, records: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, sort_keys=True) + "\n")

bst_records = [to_bst_record(row, idx) for idx, (_, row) in enumerate(dataset.iterrows())]
train_records, val_records = train_test_split(
    bst_records,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=[record["label"] for record in bst_records],
)

train_path = BST_SPLIT_DIR / "train.jsonl"
val_path = BST_SPLIT_DIR / "val.jsonl"
test_path = BST_SPLIT_DIR / "test.jsonl"
write_jsonl(train_path, train_records)
write_jsonl(val_path, val_records)
write_jsonl(test_path, val_records[: min(len(val_records), 256)])

MODEL_CONFIG = {
    "model_args": {
        "n_heads": 2,
        "k_interests": 4,
        "embed_dim": 8,
        "seq_len": MAX_HISTORY_LEN + 1,
        "intermediate_size": 32,
        "hidden_dropout_prob": 0.1,
        "attn_dropout_prob": 0.1,
        "hidden_act": "relu",
        "layer_norm_eps": 1e-12,
        "padding_idx": 0,
        "reload_model": False,
        "save_path": str(CHECKPOINT_DIR),
        "item_num": ITEM_NUM,
        "category_num": CATEGORY_NUM,
        "brand_num": BRAND_NUM,
        "price_bucket_num": PRICE_BUCKET_NUM,
        "time_bucket_num": TIME_BUCKET_NUM,
        "event_type_num": EVENT_TYPE_NUM,
    },
    "data_args": {
        "num_workers": 0,
        "shuffle": True,
        "max_history_len": MAX_HISTORY_LEN,
        "train_data_path": str(train_path),
        "val_data_path": str(val_path),
        "test_data_path": str(test_path),
        "padding_idx": 0,
    },
    "training_args": {
        "num_epochs": 10,
        "learning_rate": 0.001,
        "weight_decay": 0.00001,
        "batch_size": 256,
        "num_workers": 0,
        "ranking_ks": [1, 3, 5, 10],
    },
}

print(f"training rows: {len(train_records):,}")
print(f"validation rows: {len(val_records):,}")
print(f"train JSONL: {train_path.relative_to(ROOT)}")
print(f"validation JSONL: {val_path.relative_to(ROOT)}")
print("train label distribution:", pd.Series([record["label"] for record in train_records]).value_counts().sort_index().to_dict())
print("validation label distribution:", pd.Series([record["label"] for record in val_records]).value_counts().sort_index().to_dict())
print("sample recommenderDataset JSONL record:")
print(json.dumps(train_records[0], indent=2)[:1200])


training rows: 7,779
validation rows: 1,945
train JSONL: notebooks/data/cluster_bst_split/train.jsonl
validation JSONL: notebooks/data/cluster_bst_split/val.jsonl
train label distribution: {0: 1962, 1: 5817}
validation label distribution: {0: 491, 1: 1454}
sample recommenderDataset JSONL record:
{
  "user_id": 669,
  "hist_item_id": [
    3712,
    3713,
    3714,
    3715,
    3716
  ],
  "hist_event_type": [
    2,
    1,
    1,
    1,
    1
  ],
  "hist_category": [
    6,
    7,
    8,
    9,
    10
  ],
  "hist_brand": [
    9,
    10,
    11,
    12,
    13
  ],
  "hist_price_bucket": [
    2,
    3,
    4,
    5,
    6
  ],
  "hist_time": [
    2,
    3,
    4,
    5,
    6
  ],
  "target_item_id": 3718,
  "target_category": 9,
  "target_brand": 11,
  "target_price_bucket": 3,
  "event_time": 1774006372,
  "label": 1
}


## Step 3. Train model on training dataset

This step uses the existing `recommenderDataset`, `Trainer`, and `BST` classes from `apps/ml-system/src/models`. Training is capped at 10 epochs.


In [14]:
import torch

from models.dataset import recommenderDataset
from models.trainer import Trainer


torch.manual_seed(RANDOM_SEED)
trainer = Trainer(MODEL_CONFIG)
train_dataset = recommenderDataset(MODEL_CONFIG["data_args"], split="train", percent=1.0)
val_dataset = recommenderDataset(MODEL_CONFIG["data_args"], split="val", percent=1.0)
train_loader = trainer.get_data_loader(train_dataset, shuffle=True)
val_loader = trainer.get_data_loader(val_dataset, shuffle=False)

print(f"dataset class: {train_dataset.__class__.__name__}")
print(f"model class: {trainer.model.__class__.__name__}")
print(f"trainer class: {trainer.__class__.__name__}")
print(f"training device: {trainer.device}")
print(f"epochs: {MODEL_CONFIG['training_args']['num_epochs']}")

history = []
for epoch in range(1, MODEL_CONFIG["training_args"]["num_epochs"] + 1):
    train_metrics = trainer.train(train_loader)
    val_metrics = trainer.evaluate(val_loader)
    trainer.scheduler.step(val_metrics["loss"])
    history.append({"epoch": epoch, "train": train_metrics, "val": val_metrics})
    print(
        f"epoch={epoch:02d} "
        f"train_loss={train_metrics['loss']:.4f} train_auc={train_metrics['auc']:.4f} "
        f"val_loss={val_metrics['loss']:.4f} val_auc={val_metrics['auc']:.4f} "
        f"val_ndcg@10={val_metrics.get('ndcg@10', 0.0):.4f}"
    )

model = trainer.model
metrics = {
    "epochs": MODEL_CONFIG["training_args"]["num_epochs"],
    "final_train": history[-1]["train"],
    "final_validation": history[-1]["val"],
    "history": history,
}


dataset class: recommenderDataset
model class: BST
trainer class: Trainer
training device: mps
epochs: 10
epoch=01 train_loss=0.5882 train_auc=0.4978 val_loss=0.5656 val_auc=0.5085 val_ndcg@10=0.7476
epoch=02 train_loss=0.5663 train_auc=0.5177 val_loss=0.5631 val_auc=0.5231 val_ndcg@10=0.7476
epoch=03 train_loss=0.5619 train_auc=0.5418 val_loss=0.5625 val_auc=0.5323 val_ndcg@10=0.7476
epoch=04 train_loss=0.5638 train_auc=0.5402 val_loss=0.5612 val_auc=0.5413 val_ndcg@10=0.7476
epoch=05 train_loss=0.5605 train_auc=0.5662 val_loss=0.5602 val_auc=0.5496 val_ndcg@10=0.7476
epoch=06 train_loss=0.5570 train_auc=0.5783 val_loss=0.5581 val_auc=0.5612 val_ndcg@10=0.7476
epoch=07 train_loss=0.5505 train_auc=0.6069 val_loss=0.5569 val_auc=0.5754 val_ndcg@10=0.7476
epoch=08 train_loss=0.5461 train_auc=0.6190 val_loss=0.5570 val_auc=0.5849 val_ndcg@10=0.7476
epoch=09 train_loss=0.5327 train_auc=0.6605 val_loss=0.5539 val_auc=0.6088 val_ndcg@10=0.7476
epoch=10 train_loss=0.5203 train_auc=0.6924 val_

## Step 4. Evaluate on validation dataset

In [16]:
validation_metrics = metrics["final_validation"]

print("validation metrics from apps/ml-system Trainer.evaluate:")
for key in sorted(validation_metrics):
    value = validation_metrics[key]
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")


validation metrics from apps/ml-system Trainer.evaluate:
  auc: 0.6363
  gauc: 0.0000
  hitrate@1: 0.7476
  hitrate@10: 0.7476
  hitrate@3: 0.7476
  hitrate@5: 0.7476
  loss: 0.5439
  mrr@1: 0.7476
  mrr@10: 0.7476
  mrr@3: 0.7476
  mrr@5: 0.7476
  ndcg@1: 0.7476
  ndcg@10: 0.7476
  ndcg@3: 0.7476
  ndcg@5: 0.7476
  num_groups: 1945
  valid_auc_groups: 0


## Step 5. Save model as `.joblib`

In [18]:
artifact = {
    "model_type": "BST",
    "dataset_class": "recommenderDataset",
    "trainer_class": "Trainer",
    "task": "binary_classification",
    "feature_columns": FEATURE_COLUMNS,
    "state_dict": {key: value.detach().cpu().numpy() for key, value in model.state_dict().items()},
    "model_config": MODEL_CONFIG,
    "metrics": metrics,
    "trained_device": str(trainer.device),
    "cluster_namespace": NAMESPACE,
    "cluster_export_pod": EXPORT_POD,
    "cluster_offline_store": "s3://recsys-feature-store/offline/user_aggregate_features",
    "training_dataset": str(CLUSTER_TRAINING_DATA_PATH.relative_to(ROOT)),
    "train_jsonl": str(train_path.relative_to(ROOT)),
    "val_jsonl": str(val_path.relative_to(ROOT)),
    "label_source_run_id": RUN_ID,
    "random_seed": RANDOM_SEED,
}
joblib.dump(artifact, MODEL_PATH)

print(f"saved model artifact: {MODEL_PATH.relative_to(ROOT)}")
print(f"artifact size bytes: {MODEL_PATH.stat().st_size:,}")
print("saved artifact keys:", sorted(artifact.keys()))


saved model artifact: notebooks/models/feast_cluster_bst_10epoch.joblib
artifact size bytes: 779,478
saved artifact keys: ['cluster_export_pod', 'cluster_namespace', 'cluster_offline_store', 'dataset_class', 'feature_columns', 'label_source_run_id', 'metrics', 'model_config', 'model_type', 'random_seed', 'state_dict', 'task', 'train_jsonl', 'trained_device', 'trainer_class', 'training_dataset', 'val_jsonl']


## Document main steps completed in this notebook

- **Step 1 - Load data from cluster offline store through Feast and merge labels:** Built the generated label table, ran Feast inside Kubernetes pod `recsys-feast-training-export`, loaded cluster offline features from `s3://recsys-feature-store/offline/user_aggregate_features`, and merged labels by `user_id`.
- **Step 2 - Split data:** Converted the Feast-exported table into JSONL records compatible with `recommenderDataset`, then used a stratified 80/20 split for training and validation sets.
- **Step 3 - Train model:** Trained the existing `BST` model through the existing `Trainer` class for exactly 10 epochs on `mps` when available, otherwise CPU.
- **Step 4 - Evaluate model:** Reported validation metrics from `Trainer.evaluate`, including AUC and ranking metrics such as NDCG/MRR/hitrate.
- **Step 5 - Save model:** Saved the BST model state, config, metrics, dataset paths, and data lineage to `notebooks/models/feast_cluster_bst_10epoch.joblib`.
